# Responsible AI Health Classifier — Fairness Audit Across Demographic Groups

**Client:** Healthcare analytics team 
**Role:** ML Engineer — Logistic Regression, SGD, Bias Analysis, Threshold Calibration 
**Result:** Discovered significantly different FN rates across gender subgroups; calibrated thresholds to enforce 80% equal sensitivity

---

## Project Overview

A healthcare analytics team needed a heart disease classifier that went beyond accuracy — they needed to understand how the model's errors were distributed across patient subgroups.

I built logistic regression from scratch with full and stochastic gradient descent, then ran a complete fairness audit: confusion matrices stratified by gender, ROC curves per subgroup, and AUC analysis. The audit revealed the model produced significantly different false negative rates for men vs. women.

Final deliverable: separate decision thresholds per gender calibrated to enforce 80% equal sensitivity across both groups.

**Stack:** Python · NumPy · scikit-learn · matplotlib · pandas

## 1. Imports & Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Data Loading & Feature Engineering

The dataset contains 8,000 de-identified patient records from NHANES with 14 clinical and lifestyle features. The target is a binary heart disease label (balanced 50/50).

**Feature engineering decisions:**
- Categorical variables (gender, race/ethnicity, alcohol, chest pain) → indicator (one-hot) encoding
- An intercept column of ones is prepended — this allows the model to learn a bias term without a separate parameter
- Numerical features are **z-score normalized** at fit time — prevents features with large scales (e.g. calories) from dominating gradient updates and dramatically improves convergence

In [ ]:
data = pd.read_csv('NHANES-heart.csv')
print(f'Dataset: {data.shape[0]} records, {data.shape[1]} columns')
data.describe()

In [ ]:
FEATURE_NAMES = [
    'intercept',
    'gender_female',
    're_hispanic', 're_white', 're_black', 're_asian',
    'chest_pain', 'drink_alcohol',
    'age', 'blood_cholesterol', 'BMI',
    'blood_pressure_sys', 'diastolic_bp', 'calories', 'family_income',
]

data_fets = np.stack([
    np.ones(len(data)),                              # intercept
    (data['gender'] == 2).astype(float),             # gender_female
    ((data['race_ethnicity'] == 1) | (data['race_ethnicity'] == 2)).astype(float),
    (data['race_ethnicity'] == 3).astype(float),
    (data['race_ethnicity'] == 4).astype(float),
    (data['race_ethnicity'] == 6).astype(float),
    (data['chest_pain_ever'] == 1).astype(float),
    (data['drink_alcohol'] == 1).astype(float),
    data['age'].values,
    data['blood_cholesterol'].values,
    data['BMI'].values,
    data['blood_pressure_sys'].values,
    data['diastolic_bp'].values,
    data['calories'].values,
    data['family_income'].values,
], axis=1).astype(float)

X = data_fets
t = np.array(data['target_heart'])
print(f'Feature matrix: {X.shape}')

In [ ]:
# Train/val/test split: 5000 / 1500 / 1500
X_trainval, X_test, t_trainval, t_test = train_test_split(
    X, t, test_size=1500, random_state=RANDOM_STATE)
X_train, X_valid, t_train, t_valid = train_test_split(
    X_trainval, t_trainval, test_size=1500, random_state=RANDOM_STATE)

# Z-score normalize numerical features (columns 8+), fit on train only
NUMERICAL_START = 8
mean = X_train[:, NUMERICAL_START:].mean(axis=0)
std  = X_train[:, NUMERICAL_START:].std(axis=0)

def normalize(X):
    X_norm = X.copy()
    X_norm[:, NUMERICAL_START:] = (X[:, NUMERICAL_START:] - mean) / std
    return X_norm

X_train_norm = normalize(X_train)
X_valid_norm = normalize(X_valid)
X_test_norm  = normalize(X_test)

print(f'Train: {X_train.shape} | Val: {X_valid.shape} | Test: {X_test.shape}')

## 3. Logistic Regression — Built from Scratch

**Model:** `p(y=1 | x, w) = sigmoid(w · x)`

**Loss:** Binary cross-entropy — `L(w) = -mean[t * log(p) + (1-t) * log(1-p)]`

**Gradient:** `dL/dw = X.T @ (p - t) / N` — the gradient of cross-entropy + sigmoid simplifies cleanly to the prediction error.

In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))


def pred(w, X):
    """Predicted probability P(y=1 | x, w) for a batch of inputs."""
    return sigmoid(X @ w)


def cross_entropy_loss(w, X, t):
    """Mean binary cross-entropy loss."""
    p = pred(w, X)
    return -np.mean(t * np.log(p + 1e-12) + (1 - t) * np.log(1 - p + 1e-12))


def accuracy(w, X, t):
    """Classification accuracy at threshold 0.5."""
    return np.mean((pred(w, X) >= 0.5).astype(int) == t)

## 4. Full Batch Gradient Descent

In [ ]:
def solve_via_gradient_descent(alpha=0.0025, niter=1000,
                               X_train=X_train_norm, t_train=t_train,
                               X_valid=X_valid_norm, t_valid=t_valid,
                               w_init=None, plot=True, label_suffix=''):
    """
    Train logistic regression via full batch gradient descent.

    Parameters
    ----------
    alpha        : float — learning rate
    niter        : int — number of gradient steps
    X_train/valid: normalized feature matrices
    t_train/valid: target label vectors
    w_init       : optional weight initialization
    plot         : bool — plot loss and accuracy curves

    Returns
    -------
    w : np.ndarray — trained weight vector
    """
    N, D = X_train.shape
    w = np.zeros(D) if w_init is None else w_init.copy()

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for i in range(niter):
        p = pred(w, X_train)
        grad = X_train.T @ (p - t_train) / N
        w -= alpha * grad

        train_losses.append(cross_entropy_loss(w, X_train, t_train))
        val_losses.append(cross_entropy_loss(w, X_valid, t_valid))
        train_accs.append(accuracy(w, X_train, t_train))
        val_accs.append(accuracy(w, X_valid, t_valid))

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(train_losses, label='Train', color='steelblue')
        axes[0].plot(val_losses,   label='Val',   color='coral')
        axes[0].set_title(f'Loss — Full Batch GD{label_suffix}', fontweight='bold')
        axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Cross-entropy')
        axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(train_accs, label='Train', color='steelblue')
        axes[1].plot(val_accs,   label='Val',   color='coral')
        axes[1].set_title(f'Accuracy — Full Batch GD{label_suffix}', fontweight='bold')
        axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Accuracy')
        axes[1].set_ylim(0.4, 1.0); axes[1].legend(); axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig('images/training_curves_gd.png', dpi=150, bbox_inches='tight')
        plt.show()

    print(f'Final — Train acc: {train_accs[-1]:.4f} | Val acc: {val_accs[-1]:.4f}')
    return w


w_gd = solve_via_gradient_descent(alpha=0.0025, niter=1000)

### 4.1 Impact of Feature Normalization

Training without normalization demonstrates why it matters in practice — features at different scales cause gradient descent to oscillate and converge slowly.

In [ ]:
print('Without normalization (raw features):')
w_unnorm = solve_via_gradient_descent(
    alpha=0.05, niter=2000,
    X_train=X_train, X_valid=X_valid,
    label_suffix=' (unnormalized)'
)

## 5. Stochastic Gradient Descent

SGD computes gradients on mini-batches instead of the full dataset per update. This trades computation efficiency for noisier gradient estimates — but in practice converges faster on large datasets and generalizes similarly.

**Why SGD curves are noisier than full batch GD:**
Each mini-batch gradient is an unbiased but high-variance estimate of the true gradient. The loss fluctuates between iterations because each batch sees a different random subset of data. With full batch GD, the gradient is exact at each step — the curve is monotonically smooth.

In [ ]:
def solve_via_sgd(alpha=0.0025, n_epochs=20, batch_size=100,
                  X_train=X_train_norm, t_train=t_train,
                  X_valid=X_valid_norm, t_valid=t_valid,
                  w_init=None, plot=True):
    """
    Train logistic regression via mini-batch stochastic gradient descent.

    Parameters
    ----------
    alpha      : float — learning rate
    n_epochs   : int — passes over the full training set
    batch_size : int — samples per gradient update

    Returns
    -------
    w : np.ndarray — trained weight vector
    """
    N, D = X_train.shape
    w = np.zeros(D) if w_init is None else w_init.copy()

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(n_epochs):
        perm = np.random.permutation(N)
        X_train, t_train = X_train[perm], t_train[perm]

        for start in range(0, N, batch_size):
            Xb = X_train[start : start + batch_size]
            tb = t_train[start : start + batch_size]
            p  = pred(w, Xb)
            w -= alpha * (Xb.T @ (p - tb) / len(tb))

        train_losses.append(cross_entropy_loss(w, X_train, t_train))
        val_losses.append(cross_entropy_loss(w, X_valid, t_valid))
        train_accs.append(accuracy(w, X_train, t_train))
        val_accs.append(accuracy(w, X_valid, t_valid))

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(train_losses, label='Train', color='steelblue')
        axes[0].plot(val_losses,   label='Val',   color='coral')
        axes[0].set_title('Loss — Mini-Batch SGD', fontweight='bold')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy')
        axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(train_accs, label='Train', color='steelblue')
        axes[1].plot(val_accs,   label='Val',   color='coral')
        axes[1].set_title('Accuracy — Mini-Batch SGD', fontweight='bold')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
        axes[1].set_ylim(0.4, 1.0); axes[1].legend(); axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig('images/training_curves_sgd.png', dpi=150, bbox_inches='tight')
        plt.show()

    print(f'Final — Train acc: {train_accs[-1]:.4f} | Val acc: {val_accs[-1]:.4f}')
    return w


w_sgd = solve_via_sgd(alpha=0.05, n_epochs=20, batch_size=100)

### 5.1 Runtime: SGD vs Full Batch GD

SGD is faster per epoch because each gradient update only processes `batch_size` examples rather than all N. For 40 epochs with the same number of parameter updates, SGD is significantly faster.

In [ ]:
n_epochs = 40
n_iter_equiv = n_epochs * (len(X_train_norm) // 100)  # equivalent total iterations

t0 = time.time()
solve_via_sgd(alpha=0.05, n_epochs=n_epochs, batch_size=50, plot=False)
sgd_time = time.time() - t0

t0 = time.time()
solve_via_gradient_descent(alpha=0.0025, niter=n_iter_equiv, plot=False)
gd_time = time.time() - t0

print(f'SGD ({n_epochs} epochs, batch=50):        {sgd_time:.2f}s')
print(f'Full GD ({n_iter_equiv} iterations): {gd_time:.2f}s')
print(f'Speedup: {gd_time / sgd_time:.1f}x')

## 6. sklearn Validation

Benchmarking against sklearn's `LogisticRegression` confirms our from-scratch implementation is correct. If the accuracy numbers diverge significantly, there's a bug in the gradient computation.

In [ ]:
lr = LogisticRegression(fit_intercept=False, max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_norm, t_train)

print(f'sklearn — Train: {lr.score(X_train_norm, t_train):.4f} | Val: {lr.score(X_valid_norm, t_valid):.4f}')
print(f'Our GD  — Train: {accuracy(w_gd, X_train_norm, t_train):.4f} | Val: {accuracy(w_gd, X_valid_norm, t_valid):.4f}')

### 6.1 Feature Coefficients

The trained weight vector reveals which features most strongly predict heart disease:
- **High positive weight** → feature presence increases predicted risk
- **High negative weight** → feature presence decreases predicted risk
- **Near zero** → feature provides little discriminative signal

In [ ]:
coefs = list(zip(FEATURE_NAMES, lr.coef_[0]))
coefs_sorted = sorted(coefs[1:], key=lambda x: abs(x[1]), reverse=True)  # skip intercept

names  = [c[0] for c in coefs_sorted]
values = [c[1] for c in coefs_sorted]
colors = ['seagreen' if v > 0 else 'coral' for v in values]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], values[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients — Feature Importance',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient value (normalized features)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('images/feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Fairness Audit — Error Analysis by Demographic Group

Overall accuracy is a poor metric for a medical classifier. A model can achieve high accuracy while systematically failing one demographic group — leaving those patients underdiagnosed.

**Approach:** Stratify the test set by gender, plot separate confusion matrices, and compare false negative rates (FNR = missed diagnoses / total actual cases).

**Why FNR matters more than FPR in healthcare:**
- **False negative** (missed disease): patient goes untreated — potentially fatal
- **False positive** (unnecessary diagnosis): additional testing — costly but manageable

A model with equal accuracy across groups can still have dramatically unequal FNRs.

In [ ]:
def plot_confusion_matrix_group(X, t, lr, group='Everyone', ax=None):
    """
    Plot a confusion matrix for a patient subgroup.

    Parameters
    ----------
    X     : feature matrix for the subgroup
    t     : true labels for the subgroup
    lr    : fitted sklearn LogisticRegression
    group : str — label for the plot title
    ax    : matplotlib axes (optional)
    """
    t_pred = lr.predict(X)
    cm = confusion_matrix(t, t_pred)
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp)
    fpr = fp / (fp + tn)

    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Heart Disease'])
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{group}\nFNR: {fnr:.3f} | FPR: {fpr:.3f}', fontweight='bold')
    return fnr, fpr

In [ ]:
male   = X_test_norm[:, 1] == 0  # gender_female == 0
female = X_test_norm[:, 1] == 1  # gender_female == 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fnr_all,    fpr_all    = plot_confusion_matrix_group(X_test_norm,         t_test,        lr, 'All Patients', axes[0])
fnr_male,   fpr_male   = plot_confusion_matrix_group(X_test_norm[male],   t_test[male],  lr, 'Male',         axes[1])
fnr_female, fpr_female = plot_confusion_matrix_group(X_test_norm[female], t_test[female],lr, 'Female',       axes[2])

plt.suptitle('Confusion Matrices — Stratified by Gender', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/confusion_matrices_gender.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFalse Negative Rate (missed disease):')
print(f'  All patients: {fnr_all:.3f}')
print(f'  Male:         {fnr_male:.3f}')
print(f'  Female:       {fnr_female:.3f}')
print(f'\nFNR gap (male vs female): {abs(fnr_male - fnr_female):.3f}')

## 8. ROC Curves — Per Subgroup

ROC curves show the full sensitivity-specificity tradeoff across all possible classification thresholds. Plotting separate curves per gender reveals whether the model has fundamentally different discriminative power for each group — not just at the default 0.5 threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for mask, label, color in [
    (np.ones(len(t_test), dtype=bool), 'All patients', 'gray'),
    (male,   'Male',   'steelblue'),
    (female, 'Female', 'coral'),
]:
    probs = lr.predict_proba(X_test_norm[mask])[:, 1]
    fpr_curve, tpr_curve, _ = roc_curve(t_test[mask], probs)
    roc_auc = auc(fpr_curve, tpr_curve)
    ax.plot(fpr_curve, tpr_curve, label=f'{label} (AUC = {roc_auc:.3f})',
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax.set_title('ROC Curves by Gender — Heart Disease Classifier',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/roc_curves_gender.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Threshold Calibration — Enforcing Equal Sensitivity

A single decision threshold (default 0.5) produces different sensitivity across groups. To enforce **80% equal sensitivity (TPR)** for both genders, we find the threshold that achieves TPR ≈ 0.80 on each group's ROC curve separately.

This is a standard fairness intervention: accept higher FPRs if necessary, but ensure the model catches disease at equal rates across demographic groups.

In [ ]:
TARGET_TPR = 0.80

fig, ax = plt.subplots(figsize=(9, 6))

for mask, label, color in [(male, 'Male', 'steelblue'), (female, 'Female', 'coral')]:
    probs = lr.predict_proba(X_test_norm[mask])[:, 1]
    fpr_c, tpr_c, thresholds = roc_curve(t_test[mask], probs)

    idx = np.abs(tpr_c - TARGET_TPR).argmin()
    best_thresh = thresholds[idx]
    best_fpr    = fpr_c[idx]
    best_tpr    = tpr_c[idx]

    ax.plot(fpr_c, tpr_c, color=color, linewidth=2, label=f'{label} (AUC={auc(fpr_c,tpr_c):.3f})')
    ax.scatter([best_fpr], [best_tpr], color=color, s=120, zorder=5,
               label=f'{label} threshold={best_thresh:.3f} → TPR={best_tpr:.2f}, FPR={best_fpr:.2f}')

    print(f'{label}: threshold={best_thresh:.4f} | TPR={best_tpr:.3f} | FPR={best_fpr:.3f}')

ax.axhline(TARGET_TPR, color='gray', linestyle='--', alpha=0.6, label=f'{TARGET_TPR:.0%} sensitivity target')
ax.plot([0,1],[0,1],'k--',alpha=0.3,linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax.set_title('Threshold Calibration for Equal Sensitivity (TPR = 80%)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('images/threshold_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

| Component | Detail |
|---|---|
| Dataset | 8,000 NHANES patient records |
| Features | 15 (indicator-encoded + z-score normalized) |
| Implementation | Logistic regression from scratch — full GD + SGD |
| Validation | Matched sklearn `LogisticRegression` accuracy |

**Fairness audit findings:**
- Default 0.5 threshold produced significantly different false negative rates across gender groups
- ROC-AUC analysis confirmed the disparity is a threshold issue, not a fundamental discriminability gap
- Calibrating separate thresholds per gender enforced 80% equal sensitivity with documented FPR tradeoffs

**Key delivery:** The client received a classifier with documented bias metrics and a calibrated threshold strategy — moving from a black-box accuracy number to an auditable, defensible clinical tool.